##In this notebook, we simulate rainfall-runoff accross CAMELS-FR catchments using the interaction-driven S-GRHyMoLAP model.
##Note that the same process is applied to other diffusions. The only thing needed is to modify the diffusion formula.

# IMPORT LIBRARIES

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import random
from datetime import datetime, timedelta
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

import zipfile
import os
from scipy.optimize import minimize
from google.colab import files

In [ ]:
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


##CAMELS-DATA from my Python library.

In [ ]:
pip install aqua-fetch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.1/289.1 kB 13.8 MB/s eta 0:00:00


In [ ]:
from aqua_fetch import RainfallRunoff

#
rr = RainfallRunoff("CAMELS_FR")

/usr/local/lib/python3.12/dist-packages/aqua_fetch/rr/utils.py:127: UserWarning: netCDF4 module is not installed. Please install it to save data in netcdf format
  warnings.warn(msg, UserWarning)


downloading 6 files to /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR
downloading ADDITIONAL_LICENSES.zip
0% of 0.07 MB downloaded
100% of 0.07 MB downloaded
downloading CAMELS_FR_attributes.zip
0% of 9.88 MB downloaded
100% of 9.88 MB downloaded
downloading CAMELS_FR_geography.zip
0% of 1.45 MB downloaded
100% of 1.45 MB downloaded
downloading CAMELS_FR_time_series.zip
0% of 361.39 MB downloaded
20% of 361.39 MB downloaded
40% of 361.39 MB downloaded
60% of 361.39 MB downloaded
80% of 361.39 MB downloaded
100% of 361.39 MB downloaded
downloading README.md
0% of 0.01 MB downloaded
100% of 0.01 MB downloaded
downloading CAMELS-FR_description.ods
0% of 0.05 MB downloaded
100% of 0.05 MB downloaded
unzipping files in /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR
unzipping ADDITIONAL_LICENSES.zip to ADDITIONAL_LICENSES
unzipping CAMELS_FR_attributes.zip to CAMELS_FR_attributes
unzipping CAMELS_FR_time_series.zip to CAMELS_FR_time_series


In [ ]:
meta, ds = rr.fetch()

Read 654 stations for 22 dyn features in 49.49 seconds with 2 cpus.


In [ ]:
import xarray as xr

print(ds)              # aperçu global
#print(ds.coords)       # coordonnées (temps, variables, stations)
#print(ds['912101A'].head())  # exemple : station 5

<xarray.Dataset> Size: 2GB
Dimensions:           (time: 18993, dynamic_features: 22)
Coordinates:
  * time              (time) datetime64[ns] 152kB 1970-01-01 ... 2021-12-31
  * dynamic_features  (dynamic_features) object 176B 'q_cms_obs' ... 'airtemp...
Data variables: (12/654)
    A105003001        (time, dynamic_features) float64 3MB 1.65e+03 ... 13.4
    A107020001        (time, dynamic_features) float64 3MB nan nan ... 4.1 13.3
    A112020001        (time, dynamic_features) float64 3MB 1.04e+03 ... 13.4
    A116003002        (time, dynamic_features) float64 3MB nan nan ... 4.3 13.3
    A140202001        (time, dynamic_features) float64 3MB nan nan ... 6.8 12.4
    A202030001        (time, dynamic_features) float64 3MB nan nan ... 7.1 13.2
    ...                ...
    Y661401001        (time, dynamic_features) float64 3MB 860.0 0.441 ... 14.3
    Y781000101        (time, dynamic_features) float64 3MB nan nan ... 6.4 15.8
    Y862000101        (time, dynamic_features) float64 3MB 

In [ ]:
print(ds.dynamic_features.values)

['q_cms_obs' 'q_mm_obs' 'tsd_val_s' 'tsd_val_q' 'tsd_val_m' 'tsd_val_c'
 'tsd_val_i' 'pcp_mm' 'pcp_mm_solfrac' 'airtemp_C_mean' 'pet_mm_ou'
 'pet_mm_pe' 'pet_mm_pm' 'windspeed_mps' 'spechum_gkg' 'lwdownrad_wm2'
 'solrad_wm2' 'tsd_swi_gr' 'tsd_swi_isba' 'tsd_swe_isba' 'airtemp_C_min'
 'airtemp_C_max']


In [ ]:
# Full period
print(ds["time"].values[0], ds["time"].values[-1])

1970-01-01T00:00:00.000000000 2021-12-31T00:00:00.000000000


#Period used

In [ ]:
ds_recent = ds.sel(time=slice("2000-01-01", "2021-12-31"))

# List of all stations
all_stations = list(ds_recent.keys())

# Selection of features of interest
features = ["q_mm_obs", "pcp_mm", "pet_mm_pm"]

# Count stations without NaN
valid_stations = []

for station_id in all_stations:
    subset = ds_recent[station_id].sel(dynamic_features=features)
    df = subset.to_dataframe().reset_index().pivot(
        index="time", columns="dynamic_features", values=station_id
    )

    if df[features].isna().sum().max() == 0:  # no missing values
        valid_stations.append(station_id)

#Metrics

In [ ]:
from numba import njit

# ============================================================
# PREDICTION INTERVAL COVERAGE PROBABILITY (PICP)
# ============================================================

@njit
def PICP(Q, L, U):

    inside = 0
    count = 0

    for i in range(len(Q)):

        if (not np.isnan(Q[i]) and
            not np.isnan(L[i]) and
            not np.isnan(U[i])):

            if L[i] <= Q[i] <= U[i]:
                inside += 1

            count += 1

    if count == 0:
        return np.nan

    return inside / count


# ============================================================
# MEAN PREDICTION INTERVAL WIDTH (MPIW)
# ============================================================

@njit
def MPIW(Q, L, U):

    width = 0.0
    count = 0

    for i in range(len(Q)):

        if (not np.isnan(Q[i]) and
            not np.isnan(L[i]) and
            not np.isnan(U[i])):

            width += (U[i] - L[i])

            count += 1

    if count == 0:
        return np.nan

    return width / count

# ============================================================
# INTERVAL SCORE
# ============================================================

@njit
def IntervalScore(Q, L, U):

    loss = 0.0
    count = 0

    for i in range(len(Q)):

        if (not np.isnan(Q[i]) and
            not np.isnan(L[i]) and
            not np.isnan(U[i])):

            width = U[i]-L[i]

            under = max(L[i]-Q[i],0.0)
            over  = max(Q[i]-U[i],0.0)

            loss += width + (2.0/alpha)*(under+over)

            count += 1

    if count == 0:
        return np.nan

    return loss/count


@njit
def NSE(obs, sim):
    n = len(obs)
    mean_obs = 0.0
    count = 0

    for i in range(n):
        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            mean_obs += obs[i]
            count += 1

    if count == 0:
        return np.nan

    mean_obs /= count

    num = 0.0
    den = 0.0

    for i in range(n):
        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            num += (sim[i] - obs[i]) ** 2
            den += (obs[i] - mean_obs) ** 2

    if den == 0:
        return np.nan

    return 1.0 - num / den


@njit
def RMSE(obs, sim):
    n = len(obs)
    mse = 0.0
    count = 0

    for i in range(n):
        if not np.isnan(obs[i]) and not np.isnan(sim[i]):
            diff = sim[i] - obs[i]
            mse += diff * diff
            count += 1

    if count == 0:
        return np.nan

    return np.sqrt(mse / count)

GRHyMoLAP

In [ ]:
# ============================================================
# PERCOLATION
# ============================================================

@njit
def Percolation(Pn, En, X1):

    n = len(Pn)

    S = np.zeros(n)
    Perc = np.zeros(n)

    S[0] = X1 / 2.0

    ratio = (4.0 / 9.0) * (S[0] / X1)
    Perc[0] = S[0] * (1 - (1 + ratio**4)**(-0.25))

    for i in range(1, n):

        temp = (S[i-1] / X1)**2

        frac = Pn[i] / X1
        Ps = X1 * (1-temp) * np.tanh(frac) / (
            1 + (S[i-1]/X1) * np.tanh(frac)
        )

        frac = En[i] / X1
        Es = S[i-1] * (2-S[i-1]/X1) * np.tanh(frac) / (
            1 + (1-S[i-1]/X1) * np.tanh(frac)
        )

        S[i] = S[i-1] + Ps - Es

        ratio = (4.0/9.0)*(S[i]/X1)
        Perc[i] = S[i]*(1-(1+ratio**4)**(-0.25))

        S[i] -= Perc[i]

    return Perc


# ============================================================
# DETERMINISTIC MODEL
# ============================================================

@njit
def Deterministic_Model(params4, Q0, Pn, En):

    MU, LAMBDA, X1, GAMMA = params4

    N = len(Pn)

    Perc = Percolation(Pn, En, X1)

    Q = np.zeros(N)
    Q[0] = Q0

    for t in range(N-1):

        Q[t+1] = (
            Q[t]
            - (MU/LAMBDA)*(Q[t])**(2*MU-1)
            + GAMMA*Perc[t+1]*Pn[t+1]
        )

        if Q[t+1] < 0:
            Q[t+1] = 0.0

    return Q


# ============================================================
# STOCHASTIC MODEL
# (Common Random Numbers)
# ============================================================

@njit
def SDE_Model(params, Q0, Pn, En, Noise):

    MU, LAMBDA, X1, GAMMA, b = params

    N = len(Pn)

    N_MC = Noise.shape[0]

    Perc = Percolation(Pn, En, X1)

    Q_ens = np.zeros((N_MC, N))

    for m in range(N_MC):

        Q = np.zeros(N)
        Q[0] = Q0

        for t in range(N-1):

            drift = (
                Q[t]
                - (MU/LAMBDA)*(Q[t])**(2*MU-1)
                + GAMMA*Perc[t+1]*Pn[t+1]
            )

            sigma = b * Perc[t+1]*Pn[t+1]

            Q[t+1] = drift + sigma * Noise[m, t]

            if Q[t+1] < 0:
                Q[t+1] = 0.0

        Q_ens[m] = Q

    return Q_ens


# ============================================================
# DETERMINISTIC OBJECTIVE
# ============================================================

def objective_det(params4, Q0, Pn, En, Qobs):

    Q_sim = Deterministic_Model(
        params4,
        Q0,
        Pn,
        En
    )

    mask = ~np.isnan(Qobs)

    return np.mean((Qobs[mask]-Q_sim[mask])**2)


# ============================================================
# STOCHASTIC OBJECTIVE
# (5 parameters optimized simultaneously)
# ============================================================

def objective(params, Q0, Pn, En, Qobs, Noise):

    Q_ens = SDE_Model(
        params,
        Q0,
        Pn,
        En,
        Noise
    )

    Q5 = np.percentile(Q_ens,5,axis=0)
    Q95 = np.percentile(Q_ens,95,axis=0)

    Q5 = np.maximum(Q5,0.0)

    score = IntervalScore(Qobs,Q5,Q95)

    if np.isnan(score):
        return np.inf

    return score


In [ ]:
from numba import njit
import numpy as np
from scipy.optimize import minimize

# ============================================================
# PARAMETERS
# ============================================================

N_MC_CAL = 250      # Monte Carlo realizations for calibration
N_MC_PRED = 1000    # Monte Carlo realizations for final prediction

alpha = 0.1

b1_ratio = 0.6
max_missing_ratio = 0.05

results = {}
stations = all_stations

for i, station_id in enumerate(stations, start=1):

    print(f"\n=== Station {station_id} ===, Number = {i}")

    Q_obs = ds_recent[station_id].sel(dynamic_features="q_mm_obs").to_numpy()
    P = ds_recent[station_id].sel(dynamic_features="pcp_mm").to_numpy()
    PET = ds_recent[station_id].sel(dynamic_features="pet_mm_pm").to_numpy()

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        continue

    missing_ratio = np.sum(np.isnan(Q_obs)) / N
    if missing_ratio > max_missing_ratio:
        print(f"Skipped ({100*missing_ratio:.2f}% missing)")
        continue

    Pn = np.maximum(0.0, P - PET)
    En = np.maximum(0.0, PET - P)

    b1 = int(N * b1_ratio)
    Q0 = Q_obs[0]

    if np.isnan(Q0):
        Q0 = 0.0

    # ============================================================
    # DETERMINISTIC CALIBRATION
    # ============================================================

    guesses = [
        [1.0, 8, 150, 0.1],
        [0.6, 2, 400, 1.0],
        [1.4, 15, 300, 0.5],
        [1.0, 10, 1000, 0.3],
        [1.8, 5, 800, 0.5]
    ]

    best = None
    best_val = np.inf

    for g in guesses:

        res = minimize(
            objective_det,
            g,
            args=(Q0, Pn[:b1], En[:b1], Q_obs[:b1]),
            method="Nelder-Mead",
            options={"maxiter": 2500}
        )

        if res.fun < best_val:
            best = res
            best_val = res.fun

    params0 = np.append(best.x, 0.2)

    # ============================================================
    # STOCHASTIC CALIBRATION
    # ============================================================

    noise = np.random.RandomState(1234).randn(N_MC_CAL, b1 - 1)

    MU0, LAMBDA0, X10, GAMMA0 = best.x

    bounds = [
        (0.5 * MU0,     1.5 * MU0),
        (0.4 * LAMBDA0, 1.6 * LAMBDA0),
        (0.4 * X10,     1.6 * X10),
        (0.3 * GAMMA0,  1.7 * GAMMA0),
        (0.01, 4.0)
    ]

    res = minimize(
        objective,
        params0,
        args=(Q0, Pn[:b1], En[:b1], Q_obs[:b1], noise),
        method="Powell",
        bounds=bounds,
        options={"maxiter": 5, "disp": False}
    )

    params = res.x

    print("Optimized parameters:", params)

    # ============================================================
    # ENSEMBLE SIMULATION
    # ============================================================

    final_noise = np.random.randn(N_MC_PRED, N - 1)

    Q_ens = SDE_Model(
        params,
        Q0,
        Pn,
        En,
        final_noise
    )

    # Ensemble statistics
    Qmean = np.mean(Q_ens, axis=0)
    Q5 = np.maximum(0.0, np.percentile(Q_ens, 5, axis=0))
    Q95 = np.percentile(Q_ens, 95, axis=0)

    del Q_ens

    # ============================================================
    # PERFORMANCE OF ENSEMBLE MEAN
    # ============================================================

    NSE_cal = NSE(Q_obs[:b1], Qmean[:b1])
    NSE_val = NSE(Q_obs[b1:], Qmean[b1:])

    RMSE_cal = RMSE(Q_obs[:b1], Qmean[:b1])
    RMSE_val = RMSE(Q_obs[b1:], Qmean[b1:])

    # ============================================================
    # UNCERTAINTY METRICS
    # ============================================================

    IS_cal = IntervalScore(Q_obs[:b1], Q5[:b1], Q95[:b1])
    IS_val = IntervalScore(Q_obs[b1:], Q5[b1:], Q95[b1:])

    PICP_cal = PICP(Q_obs[:b1], Q5[:b1], Q95[:b1])
    PICP_val = PICP(Q_obs[b1:], Q5[b1:], Q95[b1:])

    MPIW_cal = MPIW(Q_obs[:b1], Q5[:b1], Q95[:b1])
    MPIW_val = MPIW(Q_obs[b1:], Q5[b1:], Q95[b1:])

    # ============================================================
    # DISPLAY
    # ============================================================

    print(f"NSE_val = {NSE_val:.4f}")
    print(f"IS_val  = {IS_val:.4f}")

    # ============================================================
    # STORE RESULTS
    # ============================================================

    results[station_id] = {

        "params": params,

        "Qmean": Qmean,
        "Q5": Q5,
        "Q95": Q95,
        "Q_obs": Q_obs,

        "NSE_cal": NSE_cal,
        "NSE_val": NSE_val,

        "RMSE_cal": RMSE_cal,
        "RMSE_val": RMSE_val,

        "IS_cal": IS_cal,
        "IS_val": IS_val,

        "PICP_cal": PICP_cal,
        "PICP_val": PICP_val,

        "MPIW_cal": MPIW_cal,
        "MPIW_val": MPIW_val,

        "missing_ratio": missing_ratio
    }

print(f"\nFinished for {len(results)} basins.")


# ============================================================
# EXTRACT LAST PARAMETER (sigma)
# ============================================================

df_sigma = pd.DataFrame([
    {
        "station_id": station_id,
        "sigma": result["params"][-1]
    }
    for station_id, result in results.items()
])


# ============================================================
# SAVE AND DOWNLOAD
# ============================================================

df_sigma.to_csv(
    "sigma_PercPn.csv",
    index=False
)

files.download("sigma_PercPn.csv")


=== Station A105003001 ===, Number = 1
Optimized parameters: [1.20302661e+00 1.03447752e+01 4.44603124e+02 1.29497542e-01
 1.30833007e-01]
NSE_val = 0.5916
IS_val  = 1.6689

=== Station A107020001 ===, Number = 2
Optimized parameters: [1.18582485e+00 1.17276687e+01 3.94641066e+02 9.89725990e-02
 1.15817433e-01]
NSE_val = 0.5604
IS_val  = 1.5500

=== Station A112020001 ===, Number = 3
Skipped (63.53% missing)

=== Station A116003002 ===, Number = 4
Optimized parameters: [1.18981273e+00 9.96605054e+00 3.41106755e+02 1.40438201e-01
 1.38751390e-01]
NSE_val = 0.6824
IS_val  = 1.5567

=== Station A140202001 ===, Number = 5
Optimized parameters: [1.22229107e+00 1.89686969e+01 2.61392279e+02 1.97804139e-01
 2.40567899e-01]
NSE_val = 0.6600
IS_val  = 11.2190

=== Station A202030001 ===, Number = 6
Optimized parameters: [1.26578402e+00 3.01171513e+01 2.53551056e+02 8.23823090e-02
 7.82059005e-02]
NSE_val = 0.7969
IS_val  = 4.2669

=== Station A204010101 ===, Number = 7
Skipped (68.13% missing)

/tmp/ipykernel_6697/3108655249.py:130: RuntimeWarning: overflow encountered in square
  return np.mean((Qobs[mask]-Q_sim[mask])**2)


Optimized parameters: [1.95738730e+00 1.25427403e+02 6.95940205e+02 1.72208460e-02
 3.54357400e-02]
NSE_val = 0.3220
IS_val  = 0.7372

=== Station A402061001 ===, Number = 21
Optimized parameters: [  1.24923181  23.6603867  116.74162362   0.27634773   0.38115099]
NSE_val = 0.7292
IS_val  = 7.1435

=== Station A405062001 ===, Number = 22
Optimized parameters: [  1.20318388  19.97813468 171.62624063   0.19290253   0.23357802]
NSE_val = 0.7599
IS_val  = 6.6498

=== Station A414020202 ===, Number = 23
Skipped (70.28% missing)

=== Station A417301001 ===, Number = 24
Optimized parameters: [1.32100257e+00 3.93141726e+01 2.25798139e+02 1.18327510e-01
 1.40491749e-01]
NSE_val = 0.7782
IS_val  = 4.4632

=== Station A420063001 ===, Number = 25
Optimized parameters: [1.20501855e+00 1.95855628e+01 2.07755390e+02 1.58691784e-01
 1.66340400e-01]
NSE_val = 0.7986
IS_val  = 5.4355

=== Station A433301001 ===, Number = 26
Optimized parameters: [1.33536169e+00 4.21128304e+01 3.42941622e+02 5.89240164e-0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# SUMMARY OF PERFORMANCE AND UNCERTAINTY METRICS
# ============================================================

# ============================================================
# NSE
# ============================================================

NSE_cal = np.array([
    v["NSE_cal"] for v in results.values()
    if not np.isnan(v["NSE_cal"])
])

NSE_val = np.array([
    v["NSE_val"] for v in results.values()
    if not np.isnan(v["NSE_val"])
])

# ============================================================
# RMSE
# ============================================================

RMSE_cal = np.array([
    v["RMSE_cal"] for v in results.values()
    if not np.isnan(v["RMSE_cal"])
])

RMSE_val = np.array([
    v["RMSE_val"] for v in results.values()
    if not np.isnan(v["RMSE_val"])
])

# ============================================================
# INTERVAL SCORE
# ============================================================

IS_cal = np.array([
    v["IS_cal"] for v in results.values()
    if not np.isnan(v["IS_cal"])
])

IS_val = np.array([
    v["IS_val"] for v in results.values()
    if not np.isnan(v["IS_val"])
])

# ============================================================
# PICP
# ============================================================

PICP_cal = np.array([
    v["PICP_cal"] for v in results.values()
    if not np.isnan(v["PICP_cal"])
])

PICP_val = np.array([
    v["PICP_val"] for v in results.values()
    if not np.isnan(v["PICP_val"])
])

# ============================================================
# MPIW
# ============================================================

MPIW_cal = np.array([
    v["MPIW_cal"] for v in results.values()
    if not np.isnan(v["MPIW_cal"])
])

MPIW_val = np.array([
    v["MPIW_val"] for v in results.values()
    if not np.isnan(v["MPIW_val"])
])

# ============================================================
# SUMMARY FUNCTION
# ============================================================

def summarize(name, x):

    print(f"\n{name}")
    print("-" * 45)
    print(f"Mean    : {np.mean(x):.4f}")
    print(f"Median  : {np.median(x):.4f}")
    print(f"Std     : {np.std(x, ddof=1):.4f}")
    print(f"Min     : {np.min(x):.4f}")
    print(f"Max     : {np.max(x):.4f}")

# ============================================================
# PRINT RESULTS
# ============================================================

summarize("Calibration NSE", NSE_cal)
summarize("Validation NSE", NSE_val)

summarize("Calibration RMSE", RMSE_cal)
summarize("Validation RMSE", RMSE_val)

summarize("Calibration Interval Score (IS)", IS_cal)
summarize("Validation Interval Score (IS)", IS_val)

summarize("Calibration Prediction Interval Coverage Probability (PICP)", PICP_cal)
summarize("Validation Prediction Interval Coverage Probability (PICP)", PICP_val)

summarize("Calibration Mean Prediction Interval Width (MPIW)", MPIW_cal)
summarize("Validation Mean Prediction Interval Width (MPIW)", MPIW_val)


Calibration NSE
---------------------------------------------
Mean    : 0.6616
Median  : 0.6903
Std     : 0.1783
Min     : -1.4050
Max     : 0.9015

Validation NSE
---------------------------------------------
Mean    : 0.6522
Median  : 0.7096
Std     : 0.3038
Min     : -3.2471
Max     : 0.9277

Calibration RMSE
---------------------------------------------
Mean    : 0.8861
Median  : 0.6814
Std     : 0.6863
Min     : 0.0980
Max     : 5.0987

Validation RMSE
---------------------------------------------
Mean    : 0.8974
Median  : 0.6488
Std     : 0.6966
Min     : 0.0856
Max     : 5.1187

Calibration Interval Score (IS)
---------------------------------------------
Mean    : 2.4994
Median  : 1.9285
Std     : 1.9038
Min     : 0.3704
Max     : 12.4035

Validation Interval Score (IS)
---------------------------------------------
Mean    : 2.5491
Median  : 1.9415
Std     : 1.9418
Min     : 0.3877
Max     : 11.8704

Calibration Prediction Interval Coverage Probability (PICP)
----------------

In [ ]:
from google.colab import files
#
stations = [
    k for k, v in results.items()
    if (not np.isnan(v["IS_val"]) and
        not np.isnan(v["PICP_val"]) and
        not np.isnan(v["MPIW_val"]))
]

# Créer le DataFrame
metrics_val_df = pd.DataFrame({
    "Station": stations,
    "IS_val": [
        v["IS_val"] for v in results.values()
        if (not np.isnan(v["IS_val"]) and
            not np.isnan(v["PICP_val"]) and
            not np.isnan(v["MPIW_val"]))
    ],
    "PICP_val": [
        v["PICP_val"] for v in results.values()
        if (not np.isnan(v["IS_val"]) and
            not np.isnan(v["PICP_val"]) and
            not np.isnan(v["MPIW_val"]))
    ],
    "MPIW_val": [
        v["MPIW_val"] for v in results.values()
        if (not np.isnan(v["IS_val"]) and
            not np.isnan(v["PICP_val"]) and
            not np.isnan(v["MPIW_val"]))
    ]
})

# Sauvegarde locale
metrics_val_df.to_csv("2UQ_PercPn.csv", index=False)

#
#files.download("2UQ_PercPn.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>